In [0]:
dbutils.widgets.text("path","")
config_path=dbutils.widgets.get("path")

import json
import pyspark.sql.functions as F
from common_utils.logging import get_logger
from common_utils.generic_functions import safe_cast

logger=get_logger("dim_customer_gold")

logger.info("Reading gold configuation from %s",config_path)

with open(config_path,'r') as f:
    config=json.load(f)

logger.info("Gold configuration loaded successfully.")

source_config=config["source"]
target_config=config["target"]

source_name=source_config["source_name"]
source_table=source_config["source_table"]
target_table=target_config["target_table"]

logger.info("[%s] Source table %s",source_name,source_table)
logger.info("[%s] Target table %s",source_name,target_table)
# READ SOURCE TABLE
# ============================================================
logger.info("[%s] Reading customers from %s",source_name,source_table)
customer_df=spark.read.table(source_table)
logger.info("Customers table loaded successfully from source %s",source_name)

def transform_customer(df,config):
    logger.info("[%s] Starting dim_customer transformation",source_name)
    df.select("customer_id","Firs_name","Last_name","loyalty_segment","valid_from","last_updated_ts")
    df=df.withColumn("first_name",F.trim(F.col("Firs_name"))).withColumn("last_name",F.trim(F.col("Last_name")))
    df=df.withColumn("full_name",F.trim(F.concat_ws(" ",F.col("first_name"),F.col("last_name"))))
    df=df.withColumn("customer_type",F.when(F.col("loyalty_segment").isNull(),F.lit(config["transform"]["default_customer_type"]))
                     .otherwise(F.concat(F.lit("Segment_"),F.col("loyalty_segment").cast("string"))))
    df=df.withColumn("created_date",F.to_date(F.from_unixtime(F.col("valid_from").cast("long"))))
    df=df.withColumn("last_updated",F.to_timestamp("last_updated_ts"))
    df=safe_cast(df,config.get("cast_columns",{}))
    df=df.select(*config['select_columns'])
    df=(df.orderBy(F.col("last_updated_ts").desc()).dropDuplicates(["customer_id"]))
    logger.info("[%s] dim_customer transformation completed.",source_name)
    return df

logger.info("[%s] Starting dim_customer creation",source_name)
dim_customer=transform_customer(customer_df,config)
logger.info("[%s] Validating dim_customer.",source_name)
record_count=dim_customer.count()
logger.info("[%s] dim_customer recound count %s",source_name,record_count)

if record_count==0:
    raise ValueError("dim_customer contains zero records.")

duplicate_count=dim_customer.groupBy("customer_id").count().filter(F.col("count")>1).count()
if duplicate_count>0:
    raise ValueError("Duplicate customer_id deteced in dim_customer.")

display(dim_customer)
# 10. CREATE GOLD SCHEMA
logger.info("%s Creating gold schema if required",source_name)
spark.sql("create schema if not exists retaildataplatform.gold ")
# WRITE GOLD TABLE
logger.info("%s writing dim_customer to %s",source_name,target_table)
dim_customer.write.format("delta").mode("overwrite").option("overWriteSchema","true").saveAsTable(target_table)
logger.info("[%s] Gold table %s written successfully",source_name,target_table)
final_count=spark.read.table(target_table).count()
logger.info("[%s] final dim_customer record count: %s",source_name,final_count)
logger.info("[%s] dim_customer pipeline successfully completed.",source_name)